# 09.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/03-embeddings/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1

Dot product $1 \cdot 2 + 2 \cdot 1 + 2 \cdot 2 = 8$. Both vectors have length $\sqrt{1 + 4 + 4} = 3$. Cosine $= 8 / 9 = 0.889$.

In [2]:
a, b = np.array([1, 2, 2.]), np.array([2, 1, 2.])
print(round(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), 3))
print(round((a / np.linalg.norm(a)) @ (b / np.linalg.norm(b)), 3))

0.889
0.889


## Setup

In [3]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # ~90 MB download, once
print(f"device: {encoder.device}, {sum(p.numel() for p in encoder.parameters()):,} parameters, "
      f"{encoder.get_sentence_embedding_dimension()} numbers per text")

def embed(texts):
    return encoder.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)

device: mps:0, 22,713,216 parameters, 384 numbers per text


In [4]:
from download import load_retail

raw = load_retail()
names = raw["Description"].dropna().str.strip()
counts = names[names.str.len() > 3].value_counts()
not_products = ["POSTAGE", "MANUAL", "DOTCOM POSTAGE", "CARRIAGE", "BANK CHARGES",
                "DISCOUNT", "AMAZON FEE", "SAMPLES", "CRUK COMMISSION"]
# product names are written in capitals; lower-case entries are stock notes ("found", "damaged")
mostly_caps = counts.index.str.count(r"[A-Z]") >= counts.index.str.count(r"[a-z]")
products = counts[(counts >= 5) & mostly_caps & ~counts.index.isin(not_products)].index.to_series(name="name")
products = products.reset_index(drop=True)
print(f"{len(products):,} ShopWave products")
products.sample(8, random_state=1).tolist()

3,524 ShopWave products


['PEARLISED IVORY HEART LARGE',
 'SMALL LICORICE DES PINK BOWL',
 'PINK BERTIE GLASS BEAD BAG CHARM',
 'DROP DIAMANTE EARRINGS GREEN',
 'HAND WARMER OWL DESIGN',
 'NUMBER TILE COTTAGE GARDEN 9',
 'SILVER VANILLA  FLOWER CANDLE POT',
 'SET/3 RED GINGHAM ROSE STORAGE BOX']

In [5]:
E = embed(products.str.lower())

def search(query, k=5):
    scores = E @ embed([query])[0]
    best = scores.argsort()[::-1][:k]
    return [(products.iloc[i], round(float(scores[i]), 3)) for i in best]

## Exercise 2

In [6]:
for query in ["something to keep biscuits in", "something for the kitchen wall", "rainy day", "picnic", "doormat"]:
    hits = products.str.lower().str.contains(query).sum()
    print(f"\n{query!r} (keyword hits: {hits})")
    for name, score in search(query, 3):
        print(f"   {score:.3f}  {name}")


'something to keep biscuits in' (keyword hits: 0)
   0.652  MONEY BOX BISCUITS DESIGN
   0.617  GLASS JAR DIGESTIVE BISCUITS
   0.565  CHARLIE + LOLA BISCUITS TINS

'something for the kitchen wall' (keyword hits: 0)
   0.549  CREAM SWEETHEART WALL CABINET
   0.525  VINTAGE BLUE KITCHEN CABINET
   0.520  KITCHEN METAL SIGN

'rainy day' (keyword hits: 0)


   0.628  RAIN PONCHO
   0.514  RAINY LADIES BIRTHDAY CARD
   0.486  RAIN PONCHO RETROSPOT

'picnic' (keyword hits: 12)


   0.640  STRAWBERRY   PICNIC BAG
   0.578  PICNIC BASKET WICKER SMALL
   0.564  PICNIC BASKET WICKER LARGE

'doormat' (keyword hits: 28)
   0.838  DOORMAT WELCOME TO OUR HOME
   0.746  DOORMAT KEEP CALM AND COME IN
   0.741  DOORMAT FRIENDSHIP


Descriptive queries (*something to keep biscuits in*, *something for the kitchen wall*, *rainy day*) are where embeddings shine: keywords find nothing, embeddings find biscuit tins, kitchen signs and rain ponchos. For exact product types (*picnic*, *doormat*) keywords already work, and embeddings agree with them.

## Exercise 3

In [7]:
S = E @ E.T
i, j = np.where(np.triu(S, k=1) > 0.95)
pairs = pd.DataFrame({"a": products.iloc[i].values, "b": products.iloc[j].values, "cosine": S[i, j].round(3)})
same_but_spaces = pairs["a"].str.replace(" ", "") == pairs["b"].str.replace(" ", "")
print(f"{len(pairs)} pairs above 0.95; {same_but_spaces.sum()} differ only in spaces")
pairs.sample(10, random_state=0)

197 pairs above 0.95; 6 differ only in spaces


,a,b,cosine
110,PINK/WHITE CHRISTMAS TREE 60CM,PINK AND WHITE CHRISTMAS TREE 120CM,0.969
12,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE RED RETROSPOT,0.963
178,APOTHECARY MEASURING JAR,LARGE APOTHECARY MEASURING JAR,0.951
74,HANGING SPRING FLOWER EGG LARGE,HANGING SPRING FLOWER EGG SMALL,0.965
149,NUMBER TILE COTTAGE GARDEN 4,NUMBER TILE COTTAGE GARDEN 0,0.967
60,CAKE STAND VICTORIAN FILIGREE SMALL,CAKE STAND VICTORIAN FILIGREE LARGE,0.960
118,NUMBER TILE COTTAGE GARDEN 2,NUMBER TILE COTTAGE GARDEN 6,0.977
186,NUMBER TILE COTTAGE GARDEN 8,NUMBER TILE COTTAGE GARDEN 9,0.983
92,WOODLAND SMALL RED FELT HEART,WOODLAND LARGE RED FELT HEART,0.969
18,CHARLIE+LOLA PINK HOT WATER BOTTLE,CHARLIE+LOLA RED HOT WATER BOTTLE,0.954


Most pairs are **variants of one product line**: small and large, front and back door, two colours of the same notebook. A handful are the *same* product typed twice with an extra space. For a data team both matter: true duplicates split one product's sales across two rows, and variants are often worth grouping before forecasting or recommending (06.10).

## Exercise 4

In [8]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

10,000 reviews (50% positive): train 8,000 / test 2,000


,text,label
0,"Great Book, Must read! Ramon found the fabulou...",1
1,A leaker! This is the second generation of the...,0
2,Not For the Professional Cook. Henckels are we...,0
3,Arg! I'd like to see Robert Jordan put to the ...,0
4,"If I could give this minus 5 stars, I'd give i...",0


In [9]:
R = encoder.encode(test["text"].tolist(), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
scores = R @ encoder.encode(["it arrived broken and I want my money back"], normalize_embeddings=True)[0]
pd.set_option("display.max_colwidth", 200)
test.assign(similarity=scores.round(3)).nlargest(3, "similarity")[["similarity", "label", "text"]]

,similarity,label,text
9698,0.433,0,"broke. I bought this and it came in good condition, however, it wasnt in any kind of plastic wrapping or a case. It filled up fine and worked great, but after it ran out, I find out that the valve..."
2316,0.429,0,Buyer Beware. I bought this exact same grater from a dept. store and the handle immediately broke off. It is made very cheaply. Oxo wants me to pay to ship it back! Spend your money more wisely an...
2828,0.413,0,defective. Broken. I received this item broken. It is hazardous to use. UI want to return it and get a new one.


All three are complaints about damaged or faulty items and refunds, found without a single shared keyword being required. This is the retrieval step of the support chatbot in 09.10.